# 07 — Análise Comparativa Final

Consolida as métricas de todos os modelos sem re-treinar nenhum. Gera a tabela comparativa equivalente à `tab:metricas_split` do LaTeX.

| | |
|---|---|
| **Entradas** | `data/metricas_comparativo.csv` (gerado pelos notebooks 02-06) |
| **Saídas** | Tabela impressa + corpo LaTeX pronto para colar em `monografia_bcc.tex` |
| **Ordem** | 7º — depois de todos os notebooks de modelo (02-06) |


In [1]:
import sys, pathlib
_src = pathlib.Path.cwd().parent / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))
from tcc_utils import *


In [2]:
import numpy as np
import pandas as pd
import json

# Carrega métricas consolidadas
df_m = carregar_metricas()

# Ordem de apresentação (mesma do LaTeX: ARIMA, OLS, GAM, BSTS, CF)
ORDEM = ['ARIMA(4, 0, 3)', 'OLS / MQO', 'GAM (final)', 'BSTS (final)', 'Causal Forest']
df_m['_ordem'] = df_m['modelo'].map({m: i for i, m in enumerate(ORDEM)})
df_m = df_m.sort_values('_ordem').drop(columns='_ordem').reset_index(drop=True)

print("Métricas registradas:")
display(df_m[['modelo', 'r2_base', 'mae_base', 'r2_teste', 'mae_teste', 'rmse_teste']])


Métricas registradas:


,modelo,r2_base,mae_base,r2_teste,mae_teste,rmse_teste
0,"ARIMA(4, 0, 3)",NaN,0.012225,NaN,NaN,0.017085
1,OLS / MQO,0.2086,0.011093,0.2339,0.010351,0.014971
2,GAM (final),0.2946,0.010758,0.3045,0.009997,0.014265
3,BSTS (final),NaN,NaN,0.2436,0.010147,0.014877
4,Causal Forest,NaN,NaN,0.2387,0.010442,0.014925


## Tabela Comparativa — Out-of-Sample

In [3]:
# Tabela formatada (equivale à tab:metricas_split)
print("=" * 85)
print(f"{'Modelo':<22} {'R²_base':>9} {'MAE_base':>10} {'R²_teste':>10} {'MAE_teste':>12} {'RMSE_teste':>12}")
print("-" * 85)

best_r2  = df_m['r2_teste'].max()
best_mae = df_m['mae_teste'].min()

for _, row in df_m.iterrows():
    r2b   = f"{row['r2_base']:.4f}"  if pd.notna(row['r2_base'])   else   '      —'
    maeb  = f"{row['mae_base']:.6f}" if pd.notna(row['mae_base'])  else  '         —'
    r2t   = f"{row['r2_teste']:.4f}" if pd.notna(row['r2_teste'])  else  '      —'
    maet  = f"{row['mae_teste']:.6f}"if pd.notna(row['mae_teste']) else '           —'
    rmset = f"{row['rmse_teste']:.6f}"if pd.notna(row['rmse_teste'])else '           —'
    mark  = ' ★' if (pd.notna(row['r2_teste']) and row['r2_teste'] == best_r2) else ''
    print(f"  {row['modelo']:<20} {r2b:>9} {maeb:>10} {r2t:>10}{mark} {maet:>12} {rmset:>12}")

print("=" * 85)
print("★ = melhor R² out-of-sample")


Modelo                   R²_base   MAE_base   R²_teste    MAE_teste   RMSE_teste
-------------------------------------------------------------------------------------
  ARIMA(4, 0, 3)               —   0.012225          —            —     0.017085
  OLS / MQO               0.2086   0.011093     0.2339     0.010351     0.014971
  GAM (final)             0.2946   0.010758     0.3045 ★     0.009997     0.014265
  BSTS (final)                 —          —     0.2436     0.010147     0.014877
  Causal Forest                —          —     0.2387     0.010442     0.014925
★ = melhor R² out-of-sample


## Corpo LaTeX — `tab:metricas_split`

In [4]:
# Gera o corpo da tabela LaTeX pronto para colar em monografia_bcc.tex
# (substitui as linhas 1053-1056 de Overleaf Latex/monografia_bcc.tex)

print("% === Cole dentro do ambiente tabular (linhas de dados apenas) ===\n")

def fmt_val(v, decimals=4, suffix=''):
    if pd.isna(v):
        return '---'
    s = f"{v:.{decimals}f}".replace('.', '{,}')
    return s + suffix

for _, row in df_m.iterrows():
    modelo = row['modelo']
    r2b    = fmt_val(row['r2_base'])
    maeb   = fmt_val(row['mae_base'], 6)
    r2t    = fmt_val(row['r2_teste'])
    maet   = fmt_val(row['mae_teste'], 6)
    rmset  = fmt_val(row['rmse_teste'], 6)
    # Bold no melhor OOS
    if pd.notna(row['r2_teste']) and row['r2_teste'] == df_m['r2_teste'].max():
        r2t  = f"\\textbf{{{r2t}}}"
        maet = f"\\textbf{{{maet}}}"
        rmset= f"\\textbf{{{rmset}}}"
    print(f"{modelo:<22} & {r2b} & {maeb} & {r2t} & {maet} & {rmset} \\\\ \\hline")


% === Cole dentro do ambiente tabular (linhas de dados apenas) ===

ARIMA(4, 0, 3)         & --- & 0{,}012225 & --- & --- & 0{,}017085 \\ \hline
OLS / MQO              & 0{,}2086 & 0{,}011093 & 0{,}2339 & 0{,}010351 & 0{,}014971 \\ \hline
GAM (final)            & 0{,}2946 & 0{,}010758 & \textbf{0{,}3045} & \textbf{0{,}009997} & \textbf{0{,}014265} \\ \hline
BSTS (final)           & --- & --- & 0{,}2436 & 0{,}010147 & 0{,}014877 \\ \hline
Causal Forest          & --- & --- & 0{,}2387 & 0{,}010442 & 0{,}014925 \\ \hline


## Causal Forest — ATE e Interpretação

In [5]:
# Extrai e exibe os extras do Causal Forest (ATE e IC)
cf_row = df_m[df_m['modelo'] == 'Causal Forest']
if not cf_row.empty and pd.notna(cf_row.iloc[0]['extras_json']):
    extras = json.loads(cf_row.iloc[0]['extras_json'])
    print("Causal Forest — Efeito Causal Médio (ATE):")
    print(f"  ATE  = {extras.get('ate', 'N/A'):.6f}")
    print(f"  IC 95%: [{extras.get('ate_lb', 'N/A'):.6f},  {extras.get('ate_ub', 'N/A'):.6f}]")
    print(f"  Covariáveis: {extras.get('covariates', [])}")
else:
    print("Execute o notebook 06 para registrar métricas do Causal Forest.")


Causal Forest — Efeito Causal Médio (ATE):
  ATE  = -0.000002
  IC 95%: [-0.004325,  0.004322]
  Covariáveis: ['br_selic_diff', 'br_ipca', 'br_pib_ret_log', 'ipca_expectativa_diff', 'embi_brasil_diff']
